# Chapter 3.2 가우시안 판별분석 (GDA) — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter03_2_gda.ipynb)

책 본문: [3.2 가우시안 판별분석 (GDA)](https://smhanlab.com/book-ml/kor/ml1/chapter03/2.html)

이 노트북은 책 3.2절의 내용을 코드로 실행합니다.
(1) 1차원 손 계산 예의 결정 경계를 직접 밀도로 검증하고,
(2) 2차원 데이터에서 numpy로 MLE GDA를 구현해 결정 경계(직선)를
그리고, (3) 50:50과 20:80 사전확률에서 경계가 **병렬 이동**하는
효과를 시각화합니다.


In [1]:
import matplotlib
matplotlib.use("Agg")  # Colab 밖에서도 headless로 동작
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 1. 손으로 푼 1차원 예: 결정 경계 x*=5 검증

본문과 같은 데이터 — 불합격(0): [2,3,4], 합격(1): [6,7,8].
\(\mu_0=3,\ \mu_1=7,\ \sigma^2=2/3\)이고 사전확률이 0.5:0.5이므로
경계는 \(x^*=5\). 4.5/5.0/5.5에서 가우시안 밀도를 직접 계산해
본문의 표와 대조합니다."


In [2]:
mu0, mu1, sigma2 = 3.0, 7.0, 2.0/3
xs = [4.5, 5.0, 5.5]
print(f"{'x':>4}  {'P(x|y=0)':>10}  {'P(x|y=1)':>10}  예측")
for x in xs:
    d0 = stats.norm.pdf(x, mu0, np.sqrt(sigma2))
    d1 = stats.norm.pdf(x, mu1, np.sqrt(sigma2))
    pred = "1 (합격)" if d1 > d0 else ("동점(경계)" if d1 == d0 else "0 (불합격)")
    print(f"{x:>4}  {d0:>10.4f}  {d1:>10.4f}  {pred}")
# 본문의 표: 4.5→0.0452/0.0022, 5.0→0.0122/0.0122, 5.5→0.0022/0.0452
# (표의 숫자는 실제로 P(x,y)=P(x|y)·P(y) — 가우시안 밀도에 사전확률 0.5를 곱한 결합확률이다 (사후확률 비교에 필요한 값))

   x    P(x|y=0)    P(x|y=1)  예측
 4.5      0.0904      0.0045  0 (불합격)
 5.0      0.0243      0.0243  동점(경계)
 5.5      0.0045      0.0904  1 (합격)


## 2. 2차원 GDA: MLE 닫힌 형태로 파라미터 추정 + 결정 경계

본문 `fit_gda`/`gda_scores` 코드를 그대로 씁니다. 두 가우시안
클래스(공유 공분산)에서 MLE로 \(\phi, \mu_0, \mu_1, \Sigma\)를
추정하고, \(z(x)=w^Tx+b>0\)일 때 y=1인 **직선** 결정 경계를
데이터 위에 겹쳐 그립니다."


In [3]:
def fit_gda(X, y):
    """GDA 파라미터 (phi, mu0, mu1, Sigma)의 최대우도추정."""
    phi = y.mean()                                  # P(y=1) 추정
    mu0, mu1 = X[y == 0].mean(axis=0), X[y == 1].mean(axis=0)
    d0, d1 = X[y == 0] - mu0, X[y == 1] - mu1
    Sigma = (d0.T @ d0 + d1.T @ d1) / len(X)        # 두 클래스 편차 pooling
    return phi, mu0, mu1, Sigma

def gda_scores(X, phi, mu0, mu1, Sigma):
    """log P(y=1|x)/P(y=0|x) = w^Tx + b  (결정 경계는 직선!)"""
    Sigma_inv = np.linalg.inv(Sigma)
    w = Sigma_inv @ (mu1 - mu0)
    b = -0.5 * (mu1 - mu0) @ Sigma_inv @ (mu1 + mu0) + np.log(phi / (1 - phi))
    return X @ w + b, w, b

rng = np.random.default_rng(42)
mu0, mu1 = np.array([1.0, 2.0]), np.array([4.0, 5.0])
Sigma_true = np.array([[1.5, 0.8], [0.8, 1.2]])
X = np.vstack([rng.multivariate_normal(mu0, Sigma_true, 300),
               rng.multivariate_normal(mu1, Sigma_true, 300)])
y = np.concatenate([np.zeros(300), np.ones(300)])

phi, mu0_hat, mu1_hat, Sigma_hat = fit_gda(X, y)
print("추정 mu0 =", np.round(mu0_hat, 3), " (참값 [1. 2.])")
print("추정 mu1 =", np.round(mu1_hat, 3), " (참값 [4. 5.])")
scores, w, b = gda_scores(X, phi, mu0_hat, mu1_hat, Sigma_hat)
print("w =", np.round(w, 3), " b =", round(b, 3))
acc = np.mean((scores > 0).astype(int) == y)
print(f"학습 데이터 정확도: {acc:.1%}")

추정 mu0 = [1.039 2.004]  (참값 [1. 2.])
추정 mu1 = [4.038 5.026]  (참값 [4. 5.])
w = [0.706 2.478]  b = -10.499
학습 데이터 정확도: 94.0%


In [4]:
# 결정 경계 (직선)를 데이터 위에 그린다
gg = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
fig, ax = plt.subplots(figsize=(6, 5))
for c, color in [(0, "tab:blue"), (1, "tab:orange")]:
    ax.scatter(X[y == c, 0], X[y == c, 1], color=color, alpha=0.5, s=18,
               label=f"y={c} (n={int((y==c).sum())})")
# 경계: w0*x1 + w1*x2 + b = 0  →  x2 = -(w0*x1 + b)/w1
ax.plot(gg, -(w[0] * gg + b) / w[1], "k--", lw=2,
        label=f"boundary: {w[0]:.2f} x1 + {w[1]:.2f} x2 + {b:.2f} = 0")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("2D GDA: MLE estimates + linear decision boundary")
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
fig.savefig("/tmp/gda_boundary.png", dpi=110)
# 책 본문에 삽입할 SVG (로컬에서만 저장; Colab에선 아래 경로가 없으니 try로 감싼다)
try:
    fig.savefig("/home/smhan/book-ml/kor/src/images/ch03_2_gda_boundary.svg")
    print("SVG 저장: kor/src/images/ch03_2_gda_boundary.svg")
except Exception:
    print("(SVG 저장 경로 없음 — Colab 환경일 수 있음)")
plt.show()

SVG 저장: kor/src/images/ch03_2_gda_boundary.svg


## 3. 불균형 사전확률: 경계가 병렬로 이동한다

본문 "가정이 깨질 때" 절 — 같은 두 클래스 분포인데 사전확률이 50:50이면
경계는 \(x^*=5\), 불합격:합격 = 80:20이면 \(x^* \approx 5.23\)로
**소수 클래스 쪽으로 밀린다**. 2차원으로 확장해, \(\phi\)의 값만
바꾸면 결정 경계가 기울기 그대로 **병렬 이동**하는 것을 시각화합니다."


In [5]:
# 같은 데이터/공분산, 사전확률만 바꿔 결정 경계의 병렬 이동을 보인다
# 1차원 예에 해당하는 투영 방향으로 축약해 경계의 위치만 계산:
# z(x) = w^T x + b 에서 b만 -log((1-phi)/phi)만큼 바뀐다.
phis = [0.5, 0.2, 0.1]
fig, ax = plt.subplots(figsize=(6, 5))
for c, color in [(0, "tab:blue"), (1, "tab:orange")]:
    ax.scatter(X[y == c, 0], X[y == c, 1], color=color, alpha=0.25, s=14)

gg = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
# 각 phi마다: b_phi = -0.5*(mu1-mu0)^T S^-1 (mu1-mu0) + log(phi/(1-phi))
Sigma_inv = np.linalg.inv(Sigma_hat)
delta = mu1_hat - mu0_hat
base = -0.5 * delta @ Sigma_inv @ (mu1_hat + mu0_hat)
for phi in phis:
    bphi = base + np.log(phi / (1 - phi))
    ax.plot(gg, -(w[0] * gg + bphi) / w[1], "--", lw=1.8,
            label=rf"$\phi$={phi} (P(y=1) prior)")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Prior tilt: same slope, boundary shifts in parallel")
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
plt.show()

## 4. sklearn LDA와 대조

`LinearDiscriminantAnalysis`가 정확히 "공분산을 공유하는 GDA"입니다 —
같은 2차원 데이터에 sklearn LDA를 돌려 numpy GDA와 `w`의 **방향**이
같는지(정규화 후 비교) 확인합니다."


In [6]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
lda.fit(X, y)
w_sk = lda.coef_[0]
# 두 w는 같은 방향일 것 (스케일만 다를 수 있음) → 코사인 유사도로 확인
cos = w @ w_sk / (np.linalg.norm(w) * np.linalg.norm(w_sk))
print(f"numpy GDA  w = {np.round(w, 3)}")
print(f"sklearn LDA w = {np.round(w_sk, 3)}")
print(f"방향 코사인 유사도: {cos:.5f}  (≈1 → 같은 결정 경계)")
print("예측 일치율:", np.mean(lda.predict(X) == (scores > 0).astype(int)), "(1.0 → 모든 점에서 일치)")

numpy GDA  w = [0.706 2.478]
sklearn LDA w = [0.703 2.469]
방향 코사인 유사도: 1.00000  (≈1 → 같은 결정 경계)
예측 일치율: 1.0 (1.0 → 모든 점에서 일치)
